In [ ]:
# Import Necessary Libraries
import os
import shutil
import yaml
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import torch
import cv2
from pathlib import Path
from dataclasses import dataclass
from sklearn.model_selection import train_test_split


In [ ]:
# Control the GPU usage;
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("Using CPU")

In [ ]:
# Load the dataset:
DATA_ROOT = Path("../Images/MRZ_Data").resolve()
input_image_dir = DATA_ROOT / "images"
input_label_dir = DATA_ROOT / "labels"

if not input_image_dir.is_dir() or not input_label_dir.is_dir():
    raise FileNotFoundError(f"Directory not found: {input_image_dir} / {input_label_dir}")

image_stems = {p.stem for p in input_image_dir.glob("*.jpg")} | {p.stem for p in input_image_dir.glob("*.png")}
label_stems = {p.stem for p in input_label_dir.glob("*.txt")}

paired = sorted(image_stems & label_stems)          
only_img = sorted(image_stems - label_stems)
only_lbl = sorted(label_stems - image_stems)

print(f"Found the paired images and labels: {len(paired)}")
if only_img: print(f"WARNING - images without labels: {len(only_img)} -> {only_img[:5]}")
if only_lbl: print(f"WARNING - labels without images: {len(only_lbl)} -> {only_lbl[:5]}")

In [ ]:
# Train / Validation / Test  (%70 / %15 / %15):
train_stems, temp_stems = train_test_split(paired, test_size=0.3, random_state=42)
val_stems, test_stems = train_test_split(temp_stems, test_size=0.5, random_state=42)

splits = {"train": train_stems, "val": val_stems, "test": test_stems}
print(f"Training set: {len(train_stems)} images")
print(f"Validation set: {len(val_stems)} images")
print(f"Test set: {len(test_stems)} images")

In [ ]:
@dataclass
class Config:
    model_name: str = "yolo11s.pt"
    epochs: int = 250          
    batch_size: int = 16
    img_size: int = 640
    lr: float = 0.01           
    patience: int = 25         
    seed: int = 42

cfg = Config()

In [ ]:
# Read the class names from classes.txt:
classes = [c for c in (DATA_ROOT / "classes.txt").read_text(encoding="utf-8").splitlines() if c.strip()]
print("Classes:", classes)

output_directory = (DATA_ROOT / "Processed_data").resolve()

# Create output directories for images and labels:
for split in splits:
    (output_directory / "images" / split).mkdir(parents=True, exist_ok=True)
    (output_directory / "labels" / split).mkdir(parents=True, exist_ok=True)

print("Output directory:", output_directory)

In [ ]:
def write_yaml_file(classes: list[str], data_root: Path, output_path: Path) -> Path:
    data = {
        "path": str(data_root),          
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "nc": len(classes),
        "names": classes,
    }
    with open(output_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(data, f, sort_keys=False, allow_unicode=True)
    print(f"data.yaml created, {output_path}")
    print(open(output_path, encoding="utf-8").read())
    return output_path

In [ ]:
def resolve_image(stem: str) -> Path:
    for ext in (".jpg", ".png"):
        p = input_image_dir / f"{stem}{ext}"
        if p.exists():
            return p
    raise FileNotFoundError(f"Image not found: {stem}")

copied = 0
for split, stems in splits.items():
    for stem in stems:
        img = resolve_image(stem)
        shutil.copy2(img, output_directory / "images" / split / img.name)
        shutil.copy2(input_label_dir / f"{stem}.txt",
                     output_directory / "labels" / split / f"{stem}.txt")
        copied += 1

print(f"{copied} images/labels pairs copied.")
for split in splits:
    n_img = len(list((output_directory / "images" / split).glob("*.*")))
    n_lbl = len(list((output_directory / "labels" / split).glob("*.txt")))
    print(f"  {split}: {n_img} images / {n_lbl} labels")

In [ ]:
# Crate the data.yaml file for YOLO training:
yaml_path = write_yaml_file(classes, output_directory, output_directory / "data.yaml")

In [ ]:
# Train the YOLO model using the ultralytics library:
from ultralytics import YOLO

model = YOLO(cfg.model_name)   
results = model.train(
    data=str(yaml_path),
    epochs=cfg.epochs,
    batch=cfg.batch_size,
    imgsz=cfg.img_size,
    lr0=cfg.lr,
    patience=cfg.patience,
    seed=cfg.seed,
    degrees=15.0,        
    perspective=0.0010,  
    shear=8.0,           
    translate=0.1, 
    hsv_h = 0.015, 
    hsv_s = 0.7, 
    hsv_v = 0.5, 
    scale = 0.5,
    device=0 if torch.cuda.is_available() else "cpu",
    project=str((DATA_ROOT.parent / "Results").resolve()),
    name="mrz_detect2",
)

In [ ]:
# Analyze the training results:

run_dir = Path("../Images/Results/mrz_detect2").resolve()
csv_path = run_dir / "results.csv"
print("Run directory:", run_dir)

df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip()
epochs = df["epoch"]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

axes[0, 0].plot(epochs, df["train/box_loss"], label="train box_loss")
axes[0, 0].plot(epochs, df["val/box_loss"],   label="val box_loss")
axes[0, 0].set_title("Box Loss"); axes[0, 0].legend(); axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(epochs, df["train/cls_loss"], label="train cls_loss")
axes[0, 1].plot(epochs, df["val/cls_loss"],   label="val cls_loss")
axes[0, 1].set_title("Cls Loss"); axes[0, 1].legend(); axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(epochs, df["train/dfl_loss"], label="train dfl_loss")
axes[1, 0].plot(epochs, df["val/dfl_loss"],   label="val dfl_loss")
axes[1, 0].set_title("DFL Loss"); axes[1, 0].legend(); axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(epochs, df["metrics/mAP50(B)"],    label="mAP50")
axes[1, 1].plot(epochs, df["metrics/mAP50-95(B)"], label="mAP50-95")
axes[1, 1].set_title("Validation mAP"); axes[1, 1].legend(); axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

best_ep = df["metrics/mAP50-95(B)"].idxmax()
print(f"\nThe best Epoch: {int(df.loc[best_ep, 'epoch'])}  "
      f"(mAP50-95 = {df.loc[best_ep, 'metrics/mAP50-95(B)']:.4f})")
print(f"Final epoch: {int(epochs.iloc[-1])}")

gap_box = df["val/box_loss"].iloc[-1] - df["train/box_loss"].iloc[-1]
print(f"\nFinal epoch box_loss farkı (val - train): {gap_box:+.4f}")

val_min_ep = df["val/box_loss"].idxmin()
print(f"val/box_loss minimum epoch {int(df.loc[val_min_ep, 'epoch'])}")


In [ ]:
# Test Evaluation & Sample Predictions:
from ultralytics import YOLO
from pathlib import Path
import matplotlib.pyplot as plt
import random

run_dir = Path("../Images/Results/mrz_detect2").resolve()
best_weights = run_dir / "weights" / "best.pt"
print("The best weights:", best_weights)

best_model = YOLO(str(best_weights))

metrics = best_model.val(data=str(yaml_path), split="test")
print(f"mAP50:     {metrics.box.map50:.4f}")
print(f"mAP50-95:  {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}   Recall: {metrics.box.mr:.4f}")

test_img_dir = output_directory / "images" / "test"
all_imgs = list(test_img_dir.glob("*.*"))
sample_imgs = random.sample(all_imgs, k=min(4, len(all_imgs)))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, img_path in zip(axes.ravel(), sample_imgs):
    res = best_model.predict(source=str(img_path), conf=0.25, verbose=False)[0]
    ax.imshow(res.plot()[..., ::-1])
    ax.set_title(img_path.name, fontsize=9)
    ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Evaluate a single test image and save the annotated output:
%matplotlib inline
from ultralytics import YOLO
from pathlib import Path
from PIL import Image

TEST_IMAGE = r"../Images/Test_Image/passport_test6.jpg"
CONF = 0.60

best_weights = Path("../Images/Results/mrz_detect2/weights/best.pt").resolve()
OUTPUT_DIR   = Path("../Images/Outputs").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

model = YOLO(str(best_weights))

img_path = Path(TEST_IMAGE)
assert img_path.exists(), f"Image not found: {img_path}"

# Make prediction (Don't use Ultralytics' save, we will save it ourselves)
res = model.predict(source=str(img_path), conf=CONF, save=False, verbose=False)[0]

# Annotated image
annotated = res.plot()[..., ::-1]   # BGR -> RGB
annotated_img = Image.fromarray(annotated)

display(annotated_img)

out_path = OUTPUT_DIR / f"{img_path.stem}_pred.jpg"
annotated_img.save(out_path)
print(f"Saved: {out_path}")

# Detection details
print(f"\nNumber of detected objects: {len(res.boxes)}")
for i, box in enumerate(res.boxes):
    cls_id = int(box.cls[0]); conf = float(box.conf[0])
    x1, y1, x2, y2 = box.xyxy[0].tolist()
    print(f"  [{i}] '{res.names[cls_id]}'  confidence={conf:.3f}  bounding box=({x1:.0f},{y1:.0f},{x2:.0f},{y2:.0f})")


In [ ]:
# Live MRZ detection from webcam using OpenCV:
best_weights = Path("../Images/Results/mrz_detect2/weights/best.pt").resolve()
CONF = 0.35
CAM_INDEX = 0

model = YOLO(str(best_weights))

cap = cv2.VideoCapture(CAM_INDEX, cv2.CAP_DSHOW)
if not cap.isOpened():
    raise RuntimeError(f"Camera could not be opened (index={CAM_INDEX}). Try another index.")

print("Camera opened. Select the window and press 'q' to quit.")

try:
    while True:
        ok, frame = cap.read()
        if not ok:
            print("Failed to read frame.")
            break

        res = model.predict(source=frame, conf=CONF, max_det=1, verbose=False)[0]
        annotated = res.plot()

        n = len(res.boxes)
        cv2.putText(annotated, f"MRZ: {n}", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)

        cv2.imshow("MRZ Live Detection", annotated)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
finally:
    cap.release()
    cv2.destroyAllWindows()
    print("Camera released.")
